# 12주 1차시 · 비슷한 것끼리 묶기: 군집

**학습 목표**
- 정답(불량 여부) 없이 공정 기록을 비슷한 것끼리 묶는 **k-평균 군집**을 실행한다.
- 표준화(StandardScaler)를 하기 전과 후의 결과를 비교해, 단위를 맞춰야 하는 이유를 확인한다.
- 엘보 그래프로 k 후보를 고르고, 무리별 평균·불량률로 **공정 조건 패턴**을 해석한다.

**사용 데이터**: `process.csv` — 사출 성형 공정 기록 494행 (가상 데이터).
오늘은 `temp`(금형 온도, ℃)와 `thickness`(두께, mm) 두 열로 묶고, `defect`(불량 0/1)는 **묶은 뒤 해석할 때만** 씁니다.

위에서부터 차례로 ▶ 실행하세요.

## 1. 준비: 라이브러리와 데이터 불러오기

In [ ]:
!pip -q install koreanize-matplotlib
import koreanize_matplotlib   # 그래프 한글 표시
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

DATA = 'https://raw.githubusercontent.com/Daeil-Noh/Lectures/main/ax-microdegree/ai-data-analysis/data/'
df = pd.read_csv(DATA + 'process.csv')
print(df.shape)
df[['product_id', 'temp', 'thickness', 'defect', 'defect_type']].head()

## 2. 정답 없이 데이터 보기
군집은 정답을 쓰지 않습니다. 먼저 온도와 두께만으로 점을 찍어 봅니다. 눈으로 보이는 뚜렷한 덩어리가 있나요?

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df['temp'], df['thickness'], s=12, color='gray', alpha=0.6)
plt.xlabel('온도 (℃)')
plt.ylabel('두께 (mm)')
plt.title('제품 494개 — 정답 표시 없이')
plt.show()

## 3. 표준화 없이 k-평균 (k = 3)
`KMeans` 는 **가장 가까운 중심에 배정 → 중심을 평균으로 이동** 을 반복합니다.
- `n_clusters=3`: 무리 3개
- `n_init=10`: 시작 위치를 10번 바꿔 가장 촘촘한 결과를 고름
- `random_state=0`: 매번 같은 결과가 나오도록 고정

먼저 **원래 단위(℃, mm) 그대로** 넣어 봅니다.

In [ ]:
X = df[['temp', 'thickness']]          # 묶을 변수 2개
km_raw = KMeans(n_clusters=3, n_init=10, random_state=0)
df['cluster_raw'] = km_raw.fit_predict(X)

plt.figure(figsize=(7, 4.5))
plt.scatter(df['temp'], df['thickness'], c=df['cluster_raw'], cmap='tab10', vmin=0, vmax=9, s=12)
plt.xlabel('온도 (℃)')
plt.ylabel('두께 (mm)')
plt.title('표준화 없이 k-평균 — 세로 띠로 나뉜다')
plt.show()

# 무리별 평균: 두께 평균이 모두 비슷하다 → 두께가 거의 무시됨
df.groupby('cluster_raw')[['temp', 'thickness']].mean().round(3)

온도는 몇 ℃씩, 두께는 0.0x mm씩 차이 납니다. 거리를 그대로 재면 **숫자가 큰 온도**가 거리를 거의 결정해서, 두께는 무시됩니다.

## 4. 표준화: 단위 맞추기
`StandardScaler` 는 열마다 **(값 − 평균) ÷ 표준편차** 로 바꿉니다. 바꾼 뒤에는 두 열 모두 평균 0, 표준편차 1이 됩니다.

In [ ]:
scaler = StandardScaler()
X_s = scaler.fit_transform(X)            # 표준화한 값 (넘파이 배열)

print('평균  :', scaler.mean_.round(3))     # [온도 평균, 두께 평균]
print('표준편차:', scaler.scale_.round(3))  # [온도 표준편차, 두께 표준편차]
print('표준화 후 평균:', X_s.mean(axis=0).round(3), ' 표준편차:', X_s.std(axis=0).round(3))

## 5. 표준화한 뒤 다시 k-평균 (k = 3)

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0)
df['cluster'] = km.fit_predict(X_s)

plt.figure(figsize=(7, 4.5))
plt.scatter(df['temp'], df['thickness'], c=df['cluster'], cmap='tab10', vmin=0, vmax=9, s=12)
plt.xlabel('온도 (℃)')
plt.ylabel('두께 (mm)')
plt.title('표준화 후 k-평균 — 온도와 두께가 함께 무리를 가른다')
plt.show()

## 6. k 고르기: 엘보 그래프
`inertia_` 는 **각 점에서 자기 중심까지 거리²의 합**(무리 안 흩어짐)입니다. k를 늘리면 항상 줄어듭니다.
줄어드는 폭이 갑자기 작아지는 곳(팔꿈치)이 k 후보입니다.

In [ ]:
ks = range(1, 9)
inertia = []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_s)
    inertia.append(m.inertia_)
    print(f'k={k}  흩어짐={m.inertia_:.1f}')

plt.figure(figsize=(6, 4))
plt.plot(list(ks), inertia, marker='o')
plt.xlabel('무리 개수 k')
plt.ylabel('무리 안 흩어짐 (inertia)')
plt.title('엘보 그래프')
plt.show()

이 데이터는 팔꿈치가 아주 뚜렷하지는 않습니다(3 근처). 이럴 때는 **무리에 이름을 붙이고 조치로 이어질 수 있는지**를 함께 봅니다.

## 7. 무리 해석: 공정 조건 패턴
군집에는 `defect` 를 쓰지 않았습니다. 이제 무리별로 평균 조건과 **불량률**을 붙여 봅니다.

In [ ]:
summary = df.groupby('cluster').agg(
    제품수=('temp', 'size'),
    평균온도=('temp', 'mean'),
    평균두께=('thickness', 'mean'),
    불량률=('defect', 'mean'),
)
summary['치수불량비율'] = df.groupby('cluster')['defect_type'].apply(lambda s: (s == '치수불량').mean())
print('전체 불량률:', round(df['defect'].mean(), 3))
summary.round(3)

무리 번호(0, 1, 2)의 순서는 실행 환경에 따라 달라질 수 있습니다. **번호가 아니라 평균 조건**을 보고 이름을 붙이세요.
예) “저온·표준 두께”, “고온 조건”, “얇은 제품” — 어느 무리에 불량이, 어떤 종류의 불량이 몰려 있나요?

In [ ]:
# 불량 유형까지 나눠 보기: 무리마다 어떤 불량이 많은가?
pd.crosstab(df['cluster'], df['defect_type'], normalize='index').round(3)

## 8. 직접 바꿔 보기
1. 아래 셀의 `K` 를 4로 바꿔 실행해 보세요. 무리 이름을 어떻게 붙일 수 있나요? 2로 줄이면 무엇이 묻히나요?
2. `cols` 에 `'humidity'`(습도)를 더해 3개 변수로 묶어 보세요. 불량률이 높은 무리가 어떻게 달라지나요?
3. (도전) `StandardScaler` 를 빼고(`X_try = df[cols]`) 실행하면 결과가 어떻게 달라지나요?

In [ ]:
K = 3                                  # ← 무리 개수를 바꿔 보세요 (2, 4, 5 …)
cols = ['temp', 'thickness']           # ← 예: ['temp', 'thickness', 'humidity']

X_try = StandardScaler().fit_transform(df[cols])
df['c_try'] = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(X_try)

result = df.groupby('c_try')[cols + ['defect']].mean().round(3)
result['제품수'] = df['c_try'].value_counts().sort_index()
result.sort_values('defect')

## 정리
- **k-평균**: 가까운 중심에 배정 → 중심을 평균으로 이동, 배정이 바뀌지 않을 때까지 반복.
- **단위가 다르면 표준화가 먼저** — 그렇지 않으면 숫자가 큰 변수만 보고 묶는다.
- **k는 사람이 고른다** — 엘보로 후보를 좁히고, 이름 붙이고 조치할 수 있는 k를 고른다.
- 군집 결과는 **가설**이다. “얇은 제품 무리에 치수불량이 많다”는 원인 확인(금형·설비 점검) 전의 단서.

**생성형AI에게 물어볼 만한 질문**
- “KMeans 결과에서 무리별 평균 온도·두께·불량률 표를 보고, 각 무리에 공정 관점의 이름을 붙여 줘. 표: (표 붙여넣기)” → AI가 붙인 이름이 **표의 숫자와 맞는지** 직접 확인하세요.
- “k-평균에서 n_init 과 random_state 는 각각 무슨 역할이야? 바꾸면 결과가 어떻게 달라질 수 있어?”